In [ ]:
import torch
from mptorch.quant import float_quantize

fp8_quant = lambda x: float_quantize(
    x, 3, 23, "nearest", subnormals=False, saturate=False
)

x = torch.tensor([8.0, 2.0, 5.0])
qx = fp8_quant(x)
print(qx)

In [ ]:
from mptorch import FloatingPoint

fp8e4 = FloatingPoint(exp=4, man=3, subnormals=True, saturate=False)

fp8e5 = FloatingPoint(exp=5, man=2, subnormals=True, saturate=False)

print(fp8e4.normal_max)
print(fp8e5.normal_max)

In [ ]:
from mptorch.quant import QAffineFormats, QLinear
from mptorch import FloatingPoint
from torch.testing import assert_close
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

mac_format = FloatingPoint(exp=5, man=10, subnormals=True, saturate=False)
wa_format = FloatingPoint(exp=4, man=3, subnormals=True, saturate=False)
g_format = FloatingPoint(exp=5, man=2, subnormals=True, saturate=False)
formats_q = QAffineFormats(
    fwd_mac=(mac_format,),
    bwd_mac=(mac_format,),
    weight_quant=(wa_format, "nearest"),
    grad_quant=(g_format, "nearest"),
    input_quant=(wa_format, "nearest"),
    use_scaling=True,
    scale_margin=3,
)

x = torch.randn(100, 100)
m = torch.nn.Linear(100, 200, bias=False)
qm = QLinear(100, 200, formats=formats_q, bias=False)
m = m.to(device)
qm = qm.to(device)
x = x.to(device)
qx = x.clone().detach()
m.weight.data = qm.weight.data.clone().detach()

res_m = m(x)
res_qm = qm(qx)
assert_close(res_m, res_qm, atol=1e-1, rtol=0.0)
print(res_m)
print(res_qm)

In [1]:
import torch
from mptorch.quant import binaryK_quantize
from gfloat import decode_float
from gfloat.formats import format_info_p3109
from gfloat import RoundMode, round_float
import struct


def bits_to_float(bits):
    s = struct.pack(">I", bits)
    return struct.unpack(">f", s)[0]


def float_to_bits(value):
    s = struct.pack(">f", value)
    return struct.unpack(">I", s)[0]


def assert_quant(x_arr, expected_arr, quant_fn, device):
    x = torch.tensor(x_arr, dtype=torch.float32, device=device)
    expected = torch.tensor(expected_arr, dtype=torch.float32, device=device)
    assert expected.equal(quant_fn(x))


K, P = 7, 3

binary7p3_fi = format_info_p3109(K, P)

torch.set_printoptions(precision=20)

x = torch.tensor(
    [
        2**-7 * (1.0 / 4 + 1.0 / 8),
        2**7 * (1.0 + 1.0 / 2 + 1.0 / 8),
        2.0**-5 * (1.0 + 1.0 / 2 + 1.0 / 8),
    ]
)
qx_ne = binaryK_quantize(x, K, P, rounding="nearest_even")
qx_na = binaryK_quantize(x, K, P, rounding="nearest_away")

gqx_ne = x.clone().detach()
gqx_ne.apply_(lambda x: round_float(binary7p3_fi, x, rnd=RoundMode.TiesToEven))
gqx_na = x.clone().detach()
gqx_na.apply_(lambda x: round_float(binary7p3_fi, x, rnd=RoundMode.TiesToAway))
print(x)
print(qx_ne)
print(gqx_ne)
print(qx_na)
print(gqx_na)

tensor([2.92968750000000000000e-03, 2.08000000000000000000e+02,
        5.07812500000000000000e-02])
tensor([3.90625000000000000000e-03, 1.92000000000000000000e+02,
        4.68750000000000000000e-02])
tensor([3.90625000000000000000e-03, 1.92000000000000000000e+02,
        4.68750000000000000000e-02])
tensor([0.00390625000000000000,                    inf, 0.05468750000000000000])
tensor([0.00390625000000000000,                    inf, 0.05468750000000000000])


In [ ]:
start_value = 2**-11 * (1.0 + 1.0 / 16)


increment = 0x7FFFFFFF & (1 << (22 - (P - 1) - 2))
print(bits_to_float(increment))
istart = float_to_bits(start_value)

vals = [bits_to_float(istart + i * increment) for i in range(0, 20 * 2**5)]

x = torch.tensor(vals, dtype=torch.float32)
gqx_ne = x.clone().detach()
gqx_na = x.clone().detach()
gqx_ne.apply_(lambda x: round_float(binary7p3_fi, x, rnd=RoundMode.TiesToEven))
gqx_na.apply_(lambda x: round_float(binary7p3_fi, x, rnd=RoundMode.TiesToAway))

qx_ne = binaryK_quantize(x, K, P, rounding="nearest_even")
qx_na = binaryK_quantize(x, K, P, rounding="nearest_away")
print(torch.all(gqx_ne == qx_ne))
print(torch.all(qx_na == gqx_na))

3.6734198463196485e-40
tensor(True)
tensor(0.00097656250000000000)
tensor(0.00195312500000000000)
tensor(0.00195312500000000000)
tensor(True)
